In [1]:
import gc
from pathlib import Path
import pandas as pd
import geopandas as gpd


In [2]:
GDF_PATH = Path("Data/kd_z_nevarnost.geojson")
XYZ_DIR = Path("Data_Processing/DMV0125")
OUTPUT_PATH = Path("Data/kd_visine.geojson")
XYZ_PATTERN = "*.XYZ"
XYZ_CRS = "EPSG:3794"
MAX_DISTANCE = 30.0


In [3]:
gdf = gpd.read_file(GDF_PATH)
if gdf.crs is None:
    raise ValueError(f"{GDF_PATH} has no CRS; set it before running the join.")

gdf_work = gdf.to_crs(XYZ_CRS).copy()
gdf_work['_best_distance'] = float('inf')
gdf_work['_best_x'] = pd.NA
gdf_work['_best_y'] = pd.NA
gdf_work['_best_z'] = pd.NA
left_geometries = gdf_work[['geometry']].copy()

xyz_files = sorted(XYZ_DIR.glob(XYZ_PATTERN))
if not xyz_files:
    raise FileNotFoundError(f"No XYZ files found in {XYZ_DIR.resolve()}")

print(f"Loaded {len(gdf_work):,} target features")
print(f"Found {len(xyz_files):,} XYZ files")


Loaded 31,086 target features
Found 3,258 XYZ files


In [4]:
for file_idx, xyz_file in enumerate(xyz_files, start=1):
    print(f"[{file_idx}/{len(xyz_files)}] {xyz_file}")

    df_part = pd.read_csv(
        xyz_file,
        sep=r"\s+",
        header=None,
        names=['x', 'y', 'z'],
        engine='python',
        dtype={'x': 'float64', 'y': 'float64', 'z': 'float64'},
    )

    if df_part.empty:
        continue

    heights = gpd.GeoDataFrame(
        df_part,
        geometry=gpd.points_from_xy(df_part['x'], df_part['y']),
        crs=XYZ_CRS,
    )

    joined = gpd.sjoin_nearest(
        left_geometries,
        heights[['x', 'y', 'z', 'geometry']],
        how='left',
        max_distance=MAX_DISTANCE,
        distance_col='_distance',
    )

    joined = (
        joined.reset_index(names='_left_index')
        .sort_values(['_left_index', '_distance', 'x', 'y', 'z'], kind='stable')
        .drop_duplicates(subset='_left_index', keep='first')
        .set_index('_left_index')
    )

    better_mask = joined['_distance'].notna() & (
        joined['_distance'] < gdf_work.loc[joined.index, '_best_distance']
    )

    if better_mask.any():
        better_index = joined.index[better_mask]
        gdf_work.loc[better_index, '_best_distance'] = joined.loc[better_index, '_distance'].to_numpy()
        gdf_work.loc[better_index, '_best_x'] = joined.loc[better_index, 'x'].to_numpy()
        gdf_work.loc[better_index, '_best_y'] = joined.loc[better_index, 'y'].to_numpy()
        gdf_work.loc[better_index, '_best_z'] = joined.loc[better_index, 'z'].to_numpy()

    del df_part, heights, joined
    gc.collect()


[1/3258] Data_Processing\DMV0125\VTA2308D96.XYZ
[2/3258] Data_Processing\DMV0125\VTA2309D96.XYZ
[3/3258] Data_Processing\DMV0125\VTA2310D96.XYZ
[4/3258] Data_Processing\DMV0125\VTA2318D96.XYZ
[5/3258] Data_Processing\DMV0125\VTA2319D96.XYZ
[6/3258] Data_Processing\DMV0125\VTA2320D96.XYZ
[7/3258] Data_Processing\DMV0125\VTA2329D96.XYZ
[8/3258] Data_Processing\DMV0125\VTA2330D96.XYZ
[9/3258] Data_Processing\DMV0125\VTA2439D96.XYZ
[10/3258] Data_Processing\DMV0125\VTA2440D96.XYZ
[11/3258] Data_Processing\DMV0125\VTA2449D96.XYZ
[12/3258] Data_Processing\DMV0125\VTA2450D96.XYZ
[13/3258] Data_Processing\DMV0125\VTA2505D96.XYZ
[14/3258] Data_Processing\DMV0125\VTA2506D96.XYZ
[15/3258] Data_Processing\DMV0125\VTA2507D96.XYZ
[16/3258] Data_Processing\DMV0125\VTA2508D96.XYZ
[17/3258] Data_Processing\DMV0125\VTA2509D96.XYZ
[18/3258] Data_Processing\DMV0125\VTA2510D96.XYZ
[19/3258] Data_Processing\DMV0125\VTA2516D96.XYZ
[20/3258] Data_Processing\DMV0125\VTA2517D96.XYZ
[21/3258] Data_Processing\DMV

In [8]:
result = gdf_work.copy()
result['x'] = result.pop('_best_x')
result['y'] = result.pop('_best_y')
result['z'] = result.pop('_best_z')
result['nearest_distance'] = result.pop('_best_distance')
result.loc[result['nearest_distance'] == float('inf'), 'nearest_distance'] = pd.NA
result = result.drop(columns=['index_right', 'x', 'y', 'nearest_distance'], errors='ignore')
result.to_crs(epsg=4326).to_file(OUTPUT_PATH, driver='GeoJSON')
result.columns


Index(['ESD', 'EID', 'IME', 'SINONIMI', 'OPIS', 'ZVRST', 'TIP', 'GESLA',
       'DATACIJA', 'LOKACIJAOPIS', 'OBCINA', 'ZAVOD', 'USMERITVE', 'STATUS',
       'PREDPIS', 'PREDPISHTTP', 'VELJAVNOST', 'FOTOAVTOR', 'FOTODATOTEKA',
       'POVEZAVA1', 'SPOMENIK', 'OBCINAID', 'QR', 'X', 'Y', 'PHOTOURL',
       'poplave_ocena', 'poplave', 'pozar', 'plazovi', 'regija', 'UE_UIME',
       'potres', 'geometry', 'z'],
      dtype='str')